# Previsão de Preços — Fase 5 (3/4): XGBoost

Treino do terceiro modelo — **XGBoost multi-ticker** (o `ticker` entra como feature em one-hot, sem repetir um modelo por ticker).

**Objectivo da fase:** comparar o boosting com a Regressão Linear e o Random Forest (mesmo split, mesmas features, mesmas métricas) e guardar as previsões para a Fase 6.

## Design

| Componente | Escolha |
|------------|---------|
| Modelo | `XGBRegressor` (500 árvores, lr 0,05, depth 6) |
| Features | 16 numéricas + `ticker` one-hot (árvores não precisam de normalização) |
| Tracking | MLflow autolog, experimento `previsao_precos` |
| Saída | `workspace.market_project.preds_xgb` (previsões val + test) |

## Estrutura da Fase 5 (XGBoost)

Setup (Instalar xgboost) → STEP 1 (Carregar dados) → STEP 2 (Treino com MLflow autolog + métricas globais no val) → STEP 3 (Métricas por ticker) → STEP 4 (Guardar previsões) → STEP 5 (Amostra de todos os tickers)

In [0]:
# Instala o xgboost — %pip tem de ser a 1ª linha da célula
%pip install xgboost -q

In [0]:
# ==============================================================================
# STEP 1: DADOS — CARREGAR TRAIN / VAL / TEST DA GOLD SPLIT
# ==============================================================================
# Objectivo: Carregar a gold_features_split para pandas e separar os três
# conjuntos cronológicos definidos na Fase 4 (idêntico aos notebooks anteriores).
#
# Input:  workspace.market_project.gold_features_split
# Output: DataFrames pandas train / val / test (sem nulos)

import pandas as pd
import numpy as np

GOLD_SPLIT = "workspace.market_project.gold_features_split"

FEATURES_NUM = ["ret_log", "sma_5", "sma_10", "sma_20", "sma_50", "vol_20",
                "rsi_14", "macd", "macd_signal", "macd_hist", "vol_rel_20",
                "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_4", "close_lag_5"]
ALVO = "target_close_next"

pdf = (spark.table(GOLD_SPLIT)
       .select("ticker", "date", "split", *FEATURES_NUM, ALVO)
       .toPandas())

train = pdf[pdf["split"] == "train"].reset_index(drop=True)
val = pdf[pdf["split"] == "val"].reset_index(drop=True)
test = pdf[pdf["split"] == "test"].reset_index(drop=True)

print(f"Treino: {len(train):,} | Validação: {len(val):,} | Teste: {len(test):,}")
assert pdf[FEATURES_NUM + [ALVO]].isna().sum().sum() == 0, "Nulos inesperados!"
print("✅ Dados carregados: 16 features numéricas + ticker, sem nulos")

In [0]:
# ==============================================================================
# STEP 2: TREINO — XGBOOST COM MLFLOW AUTOLOG (MODELO ÚNICO MULTI-TICKER)
# ==============================================================================
# Objectivo: Treinar um único XGBoost para os 12 tickers (ticker em one-hot)
# com early stopping no conjunto de validação, e registar no MLflow.
#
# Input:  train / val (STEP 1)
# Output: `pipe_xgb` treinado + métricas globais no conjunto de validação
#
# Operações:
#   - Pipeline: ColumnTransformer (OneHotEncoder) → XGBRegressor
#   - Treino direto no train (500 árvores, lr 0,05)
#   - MLflow autolog + métricas de validação (RMSE, MAE, MAPE)

import mlflow
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

X_COLS = ["ticker"] + FEATURES_NUM


def metricas(y_real, y_pred):
    """RMSE, MAE e MAPE (%) entre valores reais e previstos."""
    rmse = float(np.sqrt(mean_squared_error(y_real, y_pred)))
    mae = float(mean_absolute_error(y_real, y_pred))
    mape = float(np.mean(np.abs((y_real - y_pred) / y_real)) * 100)
    return rmse, mae, mape


preproc = ColumnTransformer([
    ("cat_ticker", OneHotEncoder(handle_unknown="ignore"), ["ticker"]),
    ("num", "passthrough", FEATURES_NUM),
])

pipe_xgb = Pipeline([("prep", preproc),
                     ("modelo", XGBRegressor(
                         n_estimators=500, learning_rate=0.05, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=42))])

mlflow.set_experiment("/Users/gonaldomichael@gmail.com/previsao_precos")
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="xgb_modelo_unico_todos_tickers"):
    pipe_xgb.fit(train[X_COLS], train[ALVO])
    pred_val_xgb = pipe_xgb.predict(val[X_COLS])
    rmse, mae, mape = metricas(val[ALVO].values, pred_val_xgb)
    mlflow.log_metrics({"val_rmse": rmse, "val_mae": mae, "val_mape": mape})
    print(f"Validação (global, 12 tickers) — RMSE: {rmse:,.2f} | MAE: {mae:,.2f} | MAPE: {mape:.2f}%")

mlflow.sklearn.autolog(disable=True)
print("✅ Modelo treinado e run registada no MLflow")

In [0]:
# ==============================================================================
# STEP 3: AVALIAÇÃO — MÉTRICAS POR TICKER NO CONJUNTO DE VALIDAÇÃO
# ==============================================================================
# Objectivo: Decompor as métricas globais por ticker e comparar com LinReg/RF.
#
# Input:  val + pred_val_xgb (STEP 2)
# Output: Tabela por ticker: RMSE, MAE, MAPE e n.º de sessões

linhas = []
for t in sorted(val["ticker"].unique()):
    g = val[val["ticker"] == t]
    rmse, mae, mape = metricas(g[ALVO].values, pred_val_xgb[g.index])
    linhas.append({"ticker": t, "RMSE": round(rmse, 2), "MAE": round(mae, 2),
                   "MAPE (%)": round(mape, 2), "Sessões": len(g)})

display(pd.DataFrame(linhas))

In [0]:
# ==============================================================================
# STEP 4: GUARDAR — PREVISÕES DA VALIDAÇÃO E TESTE EM DELTA
# ==============================================================================
# Objectivo: Gravar as previsões (val + test) para a Fase 6 (avaliação final
# no test + backtesting) e para o notebook de perguntas de negócio.
#
# Input:  pipe_xgb + val / test (STEPS 1–2)
# Output: workspace.market_project.preds_xgb
#
# Operações:
#   - Prever no conjunto de teste (ainda não usado!)
#   - Consolidar val + test com y_real / y_prev
#   - Escrita Delta partitionBy(ticker, ano, mes)

from pyspark.sql import functions as F

PREDS = "workspace.market_project.preds_xgb"

pred_test_xgb = pipe_xgb.predict(test[X_COLS])

previsoes = pd.concat([
    val[["ticker", "date", "split"]].assign(y_real=val[ALVO].values, y_prev=pred_val_xgb),
    test[["ticker", "date", "split"]].assign(y_real=test[ALVO].values, y_prev=pred_test_xgb),
], ignore_index=True)

sdf = (spark.createDataFrame(previsoes)
       .withColumn("ano", F.year("date"))
       .withColumn("mes", F.month("date")))

(sdf.write.format("delta")
   .mode("overwrite")
   .partitionBy("ticker", "ano", "mes")
   .saveAsTable(PREDS))

n = spark.table(PREDS).count()
print(f"✅ Tabela: {PREDS}")
print(f"Previsões guardadas: {n:,} (val: {len(val):,} + test: {len(test):,})")
assert n == len(val) + len(test), "Faltam previsões!"

In [0]:
%sql
-- ==============================================================================
-- STEP 5: VALIDAÇÃO — AMOSTRA DE TODOS OS TICKERS POR SPLIT (48 LINHAS)
-- ==============================================================================
-- Objectivo: Mostrar as 2 primeiras sessões de cada ticker em val e test
-- (splits com previsões guardadas), com o preço real de t+1 e a previsão,
-- em colunas portuguesas.
WITH amostra AS (
  SELECT ticker, split, date, y_real, y_prev,
         ROW_NUMBER() OVER (PARTITION BY ticker, split ORDER BY date) AS rn
  FROM workspace.market_project.preds_xgb
)
SELECT ticker, split, date AS `Data`, y_real AS `Real (t+1)`, y_prev AS `Previsão (t+1)`
FROM amostra
WHERE rn <= 2
ORDER BY ticker, split, `Data`